# 03 — Model Training

Train four classifiers with time-series cross-validation.

---

## Setup & Load Features

In [ ]:
import sys, warnings, time
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams.update({'figure.facecolor':'#0E1117','axes.facecolor':'#1C2130',
    'text.color':'#FAFAFA','axes.labelcolor':'#FAFAFA',
    'xtick.color':'#FAFAFA','ytick.color':'#FAFAFA'})
ACCENT = '#00FF87'

from src.data_pipeline import DataPipeline
from src.feature_engineering import FeatureEngineer
from src.model import ModelTrainer, decode_labels

import pandas as _pd
_pd.DataFrame.to_parquet = lambda self, p, **kw: self.to_csv(str(p).replace('.parquet','.csv'), index=False)

pipeline = DataPipeline('../data')
master = pipeline.build_master(save=False)
pipeline.load_raw_data()
shootouts = pipeline.shootouts

complete = master[~master['is_incomplete'] & master['result'].notna()].reset_index(drop=True)
print(f"Complete matches: {len(complete):,}")


## Build Feature Matrix

> We use the most recent 5,000 matches for training demo speed. The full pipeline is identical — just pass `complete` instead of `train_data`.

In [ ]:
# Use last 5000 matches (data-rich modern era with good feature coverage)
# For production: use complete (all ~49K)
SAMPLE_SIZE = 5000
train_data = complete.tail(SAMPLE_SIZE).reset_index(drop=True)
print(f"Training on {len(train_data):,} matches ({train_data['date'].min().year}–{train_data['date'].max().year})")

fe = FeatureEngineer(form_windows=[5, 10], master_df=master, shootouts_df=shootouts)
fe.fit(train_data)
X = fe.transform(train_data)
y = train_data['result']
weights = train_data['match_weight']

print(f"Feature matrix: {X.shape}")
print(f"Target distribution:")
print(y.value_counts(normalize=True).round(3))


## Time-Series Cross-Validation Training

In [ ]:
trainer = ModelTrainer(n_cv_splits=5, random_state=42)

print("Starting model training with TimeSeriesSplit CV...")
print("(tune_hyperparams=True runs RandomizedSearchCV on RF & GB)\n")

t0 = time.time()
cv_results = trainer.train_all(X, y, sample_weight=weights, tune_hyperparams=True)
elapsed = time.time() - t0
print(f"\nTotal training time: {elapsed:.1f}s")


## CV Results Comparison

In [ ]:
results_df = pd.DataFrame(cv_results).T.round(4)
results_df = results_df[['accuracy','f1_weighted','f1_macro','log_loss']].copy()
print("\n" + "="*60)
print("  CROSS-VALIDATION RESULTS SUMMARY")
print("="*60)
print(results_df.to_string())
print("="*60)
print(f"\n🏆 Best model: {trainer.best_model_name}  (F1-weighted={cv_results[trainer.best_model_name]['f1_weighted']:.4f})")

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
metrics = ['accuracy', 'f1_weighted', 'log_loss']
titles  = ['Accuracy', 'F1 Weighted', 'Log Loss']
palette = [ACCENT, '#FFD700', '#FF6B6B', '#87CEEB']

for ax, metric, title in zip(axes, metrics, titles):
    vals   = [cv_results[m][metric] for m in cv_results]
    labels = list(cv_results.keys())
    bars   = ax.bar(labels, vals, color=palette)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(metric)
    ax.tick_params(axis='x', rotation=25)
    best_idx = (vals.index(min(vals)) if metric == 'log_loss'
                else vals.index(max(vals)))
    bars[best_idx].set_edgecolor('white')
    bars[best_idx].set_linewidth(3)

plt.suptitle('Model Comparison — CV Metrics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()


## Save Best Model

In [ ]:
from pathlib import Path
Path('../models').mkdir(exist_ok=True)
saved_path = trainer.save_best_model(Path('../models/best_model.pkl'))
print(f"Best model saved: {saved_path}")

# Quick sanity check: predict one match
from src.model import ModelTrainer as MT
bundle = MT.load_model(Path('../models/best_model.pkl'))
print(f"Loaded model: {bundle['model_name']}")

# Build features for Brazil vs Argentina (most recent match data)
sample_row = complete[
    ((complete['home_team']=='Brazil') | (complete['home_team']=='Argentina')) &
    ((complete['away_team']=='Brazil') | (complete['away_team']=='Argentina'))
].tail(1)

if len(sample_row):
    X_row = fe.transform(sample_row)
    proba = bundle['model'].predict_proba(X_row.values)[0]
    home = sample_row.iloc[0]['home_team']
    away = sample_row.iloc[0]['away_team']
    from src.utils import format_probability_output
    pred = format_probability_output(proba, home, away)
    print(f"\nSample prediction:")
    print(f"  {pred['summary']}")
